In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import os


In [ ]:
model=YOLO("best.pt")

In [ ]:
IMAGE_FOLDER = "akshay_separate.v1i.yolov8/train/images"

In [ ]:
feature_data=[]
print("Starting feature extraction using segmentation...")

In [ ]:
for img_name in os.listdir(IMAGE_FOLDER):

    img_path = os.path.join(IMAGE_FOLDER, img_name)

    results = model(img_path)
    r = results[0]

    if r.masks is None:
        continue

    image = cv2.imread(img_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Compute gradient ONCE per image
    gradient = cv2.Laplacian(gray, cv2.CV_64F)

    masks = r.masks.data.cpu().numpy()

    for mask in masks:

        # Resize mask to match original image size
        mask = cv2.resize(mask, (gray.shape[1], gray.shape[0]))
        mask = mask > 0.5   # Convert to boolean

        mango_pixels = gray[mask]

        if len(mango_pixels) == 0:
            continue

        mean_T = np.mean(mango_pixels)
        max_T = np.max(mango_pixels)
        std_T = np.std(mango_pixels)
        var_T = np.var(mango_pixels)

        threshold = mean_T + 1.5 * std_T
        hotspot_ratio = np.sum(mango_pixels > threshold) / len(mango_pixels)

        gradient_mean = np.mean(np.abs(gradient[mask]))

        feature_data.append([
            img_name,
            mean_T,
            max_T,
            std_T,
            var_T,
            hotspot_ratio,
            gradient_mean
        ])

In [ ]:
columns = [
    "image_name",
    "mean_T",
    "max_T",
    "std_T",
    "var_T",
    "hotspot_ratio",
    "gradient_mean"
]
df = pd.DataFrame(feature_data, columns=columns)

df.to_csv("segmentation_features.csv", index=False)

print("Done.")
print("Total mango samples:", len(df))

In [ ]:
df = pd.read_csv("segmentation_features.csv")

print("Basic Statistics:")
print(df.describe())


In [ ]:
#defining the R
def assign_R(row):
    mean_T = row["mean_T"]
    std_T = row["std_T"]
    hotspot = row["hotspot_ratio"]

    # R1 - Unripe (cool & uniform)
    if mean_T < df["mean_T"].quantile(0.25) and std_T < 10:
        return 1

    # R2 - Early Ripe (warming but stable)
    elif mean_T < df["mean_T"].quantile(0.50):
        return 2

    # R3 - Properly Ripe (warm & stable)
    elif mean_T < df["mean_T"].quantile(0.75) and std_T < 20 and hotspot < 0.10:
        return 3

    # R4 - Overripe (hot & unstable)
    else:
        return 4


df["R_label"] = df.apply(assign_R, axis=1)

In [ ]:
#defing the Q
df["Q_score"] = (
    0.6 * df["hotspot_ratio"] +
    0.4 * df["gradient_mean"]
)

# Thresholds from distribution
q1 = df["Q_score"].quantile(0.25)
q2 = df["Q_score"].quantile(0.50)
q3 = df["Q_score"].quantile(0.75)

def assign_Q(x):
    if x <= q1:
        return 1   # Premium
    elif x <= q2:
        return 2   # Minor defect
    elif x <= q3:
        return 3   # Moderate defect
    else:
        return 4   # Severe defect

df["Q_label"] = df["Q_score"].apply(assign_Q)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

features = ["mean_T", "max_T", "std_T", "var_T", "hotspot_ratio", "gradient_mean"]

X = df[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

joblib.dump(scaler, "scaler.pkl")



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report, confusion_matrix

y_r = df["R_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_r, test_size=0.2, random_state=42
)

ripeness_model = LogisticRegression(
    multi_class="multinomial",
    max_iter=1000
)

ripeness_model.fit(X_train, y_train)

y_pred = ripeness_model.predict(X_test)

print("\nRipeness Classification Report:")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


joblib.dump(ripeness_model, "ripeness_model.pkl")

In [ ]:
y_q = df["Q_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_q, test_size=0.2, random_state=42
)

quality_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

quality_model.fit(X_train, y_train)

y_pred = quality_model.predict(X_test)

print("\nQuality Classification Report:")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

joblib.dump(quality_model, "quality_model.pkl")


In [ ]:

df.to_csv("final_labeled_dataset.csv", index=False)

print("\nPipeline Completed Successfully.")